In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
columns = ['DeviceType','PrimaryErrorCode','InternalErrorCode','EventText','ErrorDescription','SoftwareVersion']

In [0]:
error_code = spark.read.format("csv").option("delimiter","|").option("header",True).load("/mnt/raw/DataPatch/ErrorDescription/")

In [0]:
for column in columns:
    error_code = error_code.withColumn(column,ltrim(rtrim(col(column))))

In [0]:
final_df = error_code.select(columns)\
                    .withColumn('ErrorCode',trim(concat(col("PrimaryErrorCode"),lit("-"),col("InternalErrorCode"))))\
                    .withColumn('CreatedBy',lit("InitialDataLoad"))\
                    .withColumn('CreatedDt',lit(current_timestamp()))\
                    .withColumn('UpdatedBy',lit("InitialDataLoad"))\
                    .withColumn('UpdatedDt',lit(current_timestamp()))

In [0]:
final_df.display()

In [0]:
final_df.write.format("delta").mode("overwrite").save("/mnt/silver/ErrorDescription")

In [0]:
%sql
Create table if not exists silverzone.ErrorDescription
using delta
location  '/mnt/silver/ErrorDescription'

In [0]:
%sql
--select * from silverzone.ErrorDescription where PrimaryErrorCode = 'Z810'